In [1]:
import torch
import torch.nn as nn
import numpy as np

# 1. 5차원 멀티모달 상태 벡터 X(t) 정의
# [Power, Temperature, EM_Leakage, Clock_Jitter, Vibration]
class PhysicalGhostMonitor(nn.Module):
    def __init__(self):
        super(PhysicalGhostMonitor, self).__init__()
        # 정상 상태 다양체(Normal Operating Manifold)를 학습할 오토인코더 구조
        self.encoder = nn.Sequential(
            nn.Linear(5, 16),
            nn.ReLU(),
            nn.Linear(16, 3)  # 3차원의 잠재 공간(Latent Space / Attractor)으로 압축
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 5)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

# 시뮬레이션 데이터 생성 (정상 동작 상태)
# 실제 하드웨어에서 전력, 열, 진동이 커플링되어 흐르는 정상 궤도(Attractor) 가정
np.random.seed(42)
num_samples = 1000

# 정상 상태의 상관관계 데이터 (물리-정보 결합 흐름)
power = np.random.normal(1.2, 0.1, num_samples)
temp = power * 45.0 + np.random.normal(0, 0.5, num_samples)  # 전력과 열의 종속성
em_leak = power * 0.3 + np.random.normal(0, 0.02, num_samples)
jitter = np.random.normal(0.005, 0.001, num_samples)
vibration = np.random.normal(0.1, 0.01, num_samples)

normal_data = np.stack([power, temp, em_leak, jitter, vibration], axis=1)
normal_tensor = torch.tensor(normal_data, dtype=torch.float32)

print("정상 상태 다양체 데이터셋 구축 완료:", normal_tensor.shape)

정상 상태 다양체 데이터셋 구축 완료: torch.Size([1000, 5])


In [2]:
import torch.optim as optim

# 1. 모델 및 옵티마이저 선언
monitor = PhysicalGhostMonitor()
criterion = nn.MSELoss()  # 정상 상태의 궤도 오차를 측정할 손실함수
optimizer = optim.Adam(monitor.parameters(), lr=0.01)

# 2. 정상 상태 다양체(Normal Operating Manifold) 학습 루프 (100에포크)
print("\n--- 정상 상태 다양체 학습 시작 ---")
for epoch in range(100):
    monitor.train()
    optimizer.zero_grad()

    reconstructed, latent = monitor(normal_tensor)
    loss = criterion(reconstructed, normal_tensor) # 입력과 출력이 얼마나 똑같은지 (정상성 학습)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/100], 매니폴드 고정 오차(Loss): {loss.item():.6f}")

# 3. 인위적 AGI 공격/변조 시나리오 데이터 생성 (Anomaly Data)
# 소프트웨어 코드가 조작되어 물리적 커플링 균형(전력 대 대비 열 발생)이 깨진 상태
attack_data = normal_data.copy()
# 공격 시나리오: 전력은 그대로인데, 비정상 연산 폭주로 인해 '열(1번 인덱스)'과 '전자기파(2번 인덱스)'가 비정상 급증
attack_data[:, 1] += 15.0  # 미세 열장 왜곡
attack_data[:, 2] += 0.5   # 전자기 차폐 돌파 시도

attack_tensor = torch.tensor(attack_data, dtype=torch.float32)

# 4. 피지컬 고스트 검증 엔진 가동 (정상 vs 공격 상태 비교)
monitor.eval()
with torch.no_grad():
    # 정상 데이터 검증
    normal_pred, _ = monitor(normal_tensor)
    normal_score = torch.mean((normal_tensor - normal_pred) ** 2, dim=1)

    # 공격 데이터 검증
    attack_pred, _ = monitor(attack_tensor)
    attack_score = torch.mean((attack_tensor - attack_pred) ** 2, dim=1)

print("\n--- 피지컬 고스트 무결성 검증 결과 ---")
print(f"정상 상태의 평균 궤도 이탈 지수: {normal_score.mean().item():.6f}")
print(f"공격 상태의 평균 궤도 이탈 지수: {attack_score.mean().item():.6f}")

# 임계치(Threshold) 설정을 통한 시뮬레이션 판정
threshold = normal_score.mean().item() + (3 * normal_score.std().item()) # 3-Sigma 규칙
detected = (attack_score > threshold).sum().item()

print(f"👉 결론: 총 {num_samples}개의 물리 흐름 중 {detected}개의 변조 신호 탐지 성공! (탐지율: {detected/num_samples*100:.1f}%)")


--- 정상 상태 다양체 학습 시작 ---
Epoch [20/100], 매니폴드 고정 오차(Loss): 77.807648
Epoch [40/100], 매니폴드 고정 오차(Loss): 13.457095
Epoch [60/100], 매니폴드 고정 오차(Loss): 0.966116
Epoch [80/100], 매니폴드 고정 오차(Loss): 0.250331
Epoch [100/100], 매니폴드 고정 오차(Loss): 0.002650

--- 피지컬 고스트 무결성 검증 결과 ---
정상 상태의 평균 궤도 이탈 지수: 0.007841
공격 상태의 평균 궤도 이탈 지수: 0.066025
👉 결론: 총 1000개의 물리 흐름 중 1000개의 변조 신호 탐지 성공! (탐지율: 100.0%)
